In [0]:
from pyspark.sql import functions as F

CAT = "databricks_proyecto_jhon"
plata = spark.table(f"{CAT}.lakehouse.plata_ventas")
plata_tc = spark.table(f"{CAT}.lakehouse.plata_tipo_cambio_bcrp")

# ==========================================
# 1. DIMENSIÓN TIEMPO (Generada sin huecos)
# ==========================================
# Extraemos fecha min y max para generar un calendario continuo
rango = plata.agg(F.min("fecha").alias("ini"), F.max("fecha").alias("fin")).collect()[0]

dim_tiempo = (spark.sql(f"""
    SELECT explode(sequence(DATE '{rango['ini']}', DATE '{rango['fin']}', INTERVAL 1 DAY)) AS fecha
""")
    .withColumn("sk_tiempo", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("nombre_mes", F.date_format("fecha", "MMMM"))
    .withColumn("anio_mes", F.date_format("fecha", "yyyy-MM"))
    .withColumn("dia_semana", F.date_format("fecha", "EEEE"))
    .withColumn("es_fin_semana", F.dayofweek("fecha").isin(1, 7))
    # Nos traemos el TC de la capa plata (que ya tiene los fines de semana imputados)
    .join(plata_tc, "fecha", "left")
)
(dim_tiempo.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{CAT}.lakehouse.dim_tiempo"))

# ==========================================
# 2. DIMENSIÓN CLIENTE
# ==========================================
dim_cliente = (plata
    .select("id_cliente", "nombre_cliente", "canal", "ciudad").distinct()
    .withColumn("sk_cliente", F.col("id_cliente"))
    .withColumn("_fecha_actualizacion", F.current_timestamp()))

(dim_cliente.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{CAT}.lakehouse.dim_cliente"))

# ==========================================
# 3. DIMENSIÓN PRODUCTO
# ==========================================
dim_producto = (plata
    .select("id_producto", "nombre_producto", "categoria", "marca", 
            F.col("costo_unit_usd").alias("costo_lista_usd")).distinct()
    .withColumn("sk_producto", F.col("id_producto"))
    .withColumn("_fecha_actualizacion", F.current_timestamp()))

(dim_producto.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{CAT}.lakehouse.dim_producto"))

# ==========================================
# 4. TABLA DE HECHOS (fct_ventas)
# ==========================================
# Leemos la dimensión tiempo recién creada para asegurar consistencia
dim_tiempo_guardada = spark.table(f"{CAT}.lakehouse.dim_tiempo")

hechos = (plata
    .filter(F.col("estado") == "FACTURADA")
    # Join con Broadcast: Ideal porque la dimensión tiempo es pequeña y cabe en memoria
    .join(F.broadcast(dim_tiempo_guardada.select("fecha", "sk_tiempo", "tc_venta", "tc_es_imputado")), "fecha", "left")
    
    # CALCULOS DE NEGOCIO Y EXPOSICIÓN CAMBIARIA
    .withColumn("tc_aplicado", F.col("tc_venta"))
    .withColumn("costo_pen", F.round(F.col("costo_usd") * F.col("tc_aplicado"), 2))
    .withColumn("monto_usd", F.round(F.col("monto_pen") / F.col("tc_aplicado"), 2))
    .withColumn("margen_pen", F.round(F.col("monto_pen") - F.col("costo_pen"), 2))
    .select(
        "sk_tiempo",
        F.col("id_cliente").alias("sk_cliente"),
        F.col("id_producto").alias("sk_producto"),
        "id_venta", "cantidad",
        "monto_pen", "monto_usd", "costo_usd", "costo_pen",
        "margen_pen", "tc_aplicado", "tc_es_imputado"
    )
    .withColumn("_fecha_carga", F.current_timestamp()))

# Guardado final de los hechos
(hechos.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{CAT}.lakehouse.fct_ventas"))

print(f"✅ [ORO] dim_tiempo: {dim_tiempo.count():,} | dim_cliente: {dim_cliente.count():,} | dim_producto: {dim_producto.count():,}")
print(f"✅ [ORO] fct_ventas: {hechos.count():,} filas generadas listas para Power BI.")